# VietRAG Direction 1 — Colab T4
Run cells in order. Select **Runtime → Change runtime type → T4 GPU** first. Restricted datasets and generated artifacts are not committed to GitHub.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU runtime before continuing'
print(torch.cuda.get_device_name(0))

## 1. Clone the repository
Replace `REPO_URL` after the repository has been created on GitHub. For a private repository, use Colab's GitHub integration or a short-lived token stored in Colab Secrets.

In [ ]:
REPO_URL = 'https://github.com/Huynhnhu169/VIETRAG_Project.git'
!git clone {REPO_URL}
%cd VIETRAG_Project

In [ ]:
!python -m pip install --quiet --upgrade pip
!python -m pip install --quiet -r requirements.txt -r requirements-ml.txt

## 2. Download and prepare ViRHE4QA
Set `ACCEPT_VIRHE4QA_LICENSE=True` only after reviewing and accepting the upstream research/non-commercial terms.

In [ ]:
ACCEPT_VIRHE4QA_LICENSE = False
assert ACCEPT_VIRHE4QA_LICENSE, 'Review data/README.md, then set this to True'
!python scripts/download_data.py --accept-license
!python scripts/prepare_corpus.py --input data/raw/ViRHE4QA.zip
!python scripts/audit_data.py
!python scripts/create_splits.py --seed 42 --document-folds 5

In [ ]:
!python -m pytest -q --basetemp /tmp/vietrag-pytest

## 3. Validation baselines
P0 is CPU BM25. P1 and P2 use BGE-M3 on the T4. Do not run on the test split while selecting models or thresholds.

In [ ]:
!python scripts/evaluate_retrieval.py --config configs/experiments/P0_bm25.yaml --split validation --output experiments/colab/P0_bm25_validation
!python scripts/evaluate_retrieval.py --config configs/colab_t4.yaml --mode dense --split validation --output experiments/colab/P1_bge_m3_validation
!python scripts/evaluate_retrieval.py --config configs/colab_t4.yaml --mode hybrid_rrf --split validation --output experiments/colab/P2_hybrid_rrf_validation

## 4. Cross-encoder reranking
This loads `BAAI/bge-reranker-v2-m3` on the T4 and reranks top 20 to top 5. If CUDA runs out of memory, change `reranking.batch_size` in `configs/colab_t4_reranker.yaml` from 4 to 2.

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
!python scripts/evaluate_retrieval.py --config configs/colab_t4_reranker.yaml --mode hybrid_rrf --split validation --output experiments/colab/P4_hybrid_reranker_validation

## 5. Inspect and download reproducibility artifacts

In [ ]:
from pathlib import Path
import json
for metrics_file in sorted(Path('experiments/colab').glob('*/metrics.json')):
    result = json.loads(metrics_file.read_text(encoding='utf-8'))['aggregate']
    print(metrics_file.parent.name, result)
!zip -qr vietrag_colab_artifacts.zip experiments/colab data/interim/audit.json data/manifests
from google.colab import files
files.download('vietrag_colab_artifacts.zip')